## ReLU MLP Single-Seed Learning Dynamics

This notebook runs a standard fully connected ReLU network with a single training phase.

It focuses on one seed to inspect learning dynamics clearly (checkpoints, scatter alignment, and start/end metrics).

No dropout, normalization, or multi-phase logic is used.

In [ ]:
from pathlib import Path

from itables import show
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

%load_ext autoreload
%autoreload 2

from layer_lenses.odt import samples_reaching_node
from layer_lenses.odt import odt_leaf_ids_for_x
from layer_lenses.relu_analysis import (
    collect_start_end_metrics,
    first_layer_odt_alignment,
    masked_accuracy_by_odt_leaf,
    neuron_leaf_responsibility_from_ablation,
    plot_first_layer_odt_alignment,
    plot_layered_relu_graph,
    quadratic_snapshot_epochs,
    run_single_relu_seed,
    summarize_neuron_responsibility,
)
from layer_lenses.relu_analysis import log_loss_gradients
from layer_lenses.relu_training import checkpoint_model_from_state
from layer_lenses.relu_analysis import activation_tensor_by_layer_neuron_data

from pathlib import Path
import pickle
from typing import Any

def save_seed_result_pickle(seed_result: dict[str, Any], path: str | Path) -> Path:
    """Save seed_result to a pickle file."""
    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("wb") as f:
        pickle.dump(seed_result, f, protocol=pickle.HIGHEST_PROTOCOL)
    return out_path


def load_seed_result_pickle(path: str | Path) -> dict[str, Any]:
    """Load seed_result from a pickle file."""
    in_path = Path(path)
    with in_path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, dict):
        raise TypeError(f"Expected dict from pickle, got {type(obj).__name__}")
    return obj


In [ ]:


# Scratch-aligned defaults with a plain ReLU MLP.
DIM = 100
DEPTH = 5
N_TRAIN = 1_000_000
HIDDEN_DIMS = [100]

TOTAL_EPOCHS = 2000
LR = 2e-3
BATCH_SIZE = 1000
WEIGHT_DECAY = 0.01

SNAPSHOT_EPOCHS = range(0,TOTAL_EPOCHS,1)
print(f"Using {len(SNAPSHOT_EPOCHS)} snapshot epochs.")

SEED = 5178

seed_result = None

## Run selected seed

This is the expensive cell. It runs single-phase ReLU MLP training for the selected seed.

In [ ]:
print(f"Running seed {SEED} ...")
seed_result = run_single_relu_seed(
    SEED,
    dim=DIM,
    depth=DEPTH,
    n_train=N_TRAIN,
    hidden_dims=HIDDEN_DIMS,
    total_epochs=TOTAL_EPOCHS,
    lr=LR,
    batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY,
    snapshot_epochs=SNAPSHOT_EPOCHS,
    project_first_layer_grad_orthogonal=None,
    odt_hyperplanes = "coordinate_axes",
    bias= False,
    device="cpu"
)
print("Done")

In [ ]:

# Save
# save_path = save_seed_result_pickle(seed_result, "results/cache/seed_result_5178_5_40_40_40_1Mtrain_1000batch.pkl")
# print("Saved to:", save_path)

# Load
# seed_result = load_seed_result_pickle("results/cache/seed_result_5178.pkl")
# print(seed_result.keys())

In [ ]:
seed_result["data"]["x_train"].shape

In [ ]:
plt.plot(seed_result["out"]["epoch_losses"][:20])
out = seed_result["out"]
ckpts = out["checkpoint_snapshots"]
tree = seed_result["tree"]

## Scatter plot and gating dynamics analysis

In [ ]:
for name, param in ckpts[499].items():
    print(name, param.shape)

In [ ]:
FIRST_LAYER_NEURON_ID = [30]
ODT_INTERNAL_NODE_ID = [0,1,2]+list[int](range(15,22))
NUM_POINTS_PLOT = 20



neuron_ids = list(FIRST_LAYER_NEURON_ID)
odt_node_ids = list(ODT_INTERNAL_NODE_ID)

if not neuron_ids:
    raise ValueError("FIRST_LAYER_NEURON_ID must not be empty.")
if not odt_node_ids:
    raise ValueError("ODT_INTERNAL_NODE_ID must not be empty.")

first_layer_width = ckpts[0]["hidden_layers.0.weight"].shape[0]
internal_node_count = tree.w_list.shape[0]

for neuron_id in neuron_ids:
    if not (0 <= neuron_id < first_layer_width):
        raise ValueError(
            f"Each FIRST_LAYER_NEURON_ID must be in [0, {first_layer_width - 1}], "
            f"got {neuron_id}."
        )

for odt_node_id in odt_node_ids:
    if not (0 <= odt_node_id < internal_node_count):
        raise ValueError(
            f"Each ODT_INTERNAL_NODE_ID must be in [0, {internal_node_count - 1}], "
            f"got {odt_node_id}."
        )

epochs = sorted(ckpts.keys())[:NUM_POINTS_PLOT]

odt_vectors = {
    odt_node_id: torch.from_numpy(tree.w_list[odt_node_id]).float()
    for odt_node_id in odt_node_ids
}
odt_norms = {
    odt_node_id: float(torch.norm(vector).clamp_min(1e-12))
    for odt_node_id, vector in odt_vectors.items()
}

fig, axes = plt.subplots(
    nrows=len(neuron_ids),
    ncols=1,
    figsize=(8, 4.5 * len(neuron_ids)),
    sharex=True,
    squeeze=False,
)
axes = axes[:, 0]

for ax, neuron_id in zip(axes, neuron_ids):
    neuron_norms = []
    normalized_inner_products = {
        odt_node_id: [] for odt_node_id in odt_node_ids
    }

    for epoch in epochs:
        w = (
            ckpts[epoch]["hidden_layers.0.weight"][neuron_id]
            .detach()
            .cpu()
            .float()
        )
        w_norm = float(torch.norm(w).clamp_min(1e-12))
        neuron_norms.append(w_norm)

        for odt_node_id in odt_node_ids:
            cosine = float(
                torch.dot(w, odt_vectors[odt_node_id])
                / (w_norm * odt_norms[odt_node_id])
            )
            normalized_inner_products[odt_node_id].append(cosine)

    for odt_node_id in odt_node_ids:
        label = (
            "normalized inner product (cosine)"
            if len(odt_node_ids) == 1
            else f"ODT node {odt_node_id} cosine"
        )
        ax.plot(
            epochs,
            normalized_inner_products[odt_node_id],
            linewidth=2.0,
            label=label,
        )

    ax.plot(
        epochs,
        neuron_norms,
        linewidth=2.0,
        label="neuron norm",
    )
    ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
    ax.set_ylabel("Value")
    ax.set_title(
        "First-layer neuron vs ODT node over checkpoints\n"
        f"neuron={neuron_id}, odt_nodes={odt_node_ids}"
    )
    ax.grid(alpha=0.25)
    ax.legend()

axes[-1].set_xlabel("Checkpoint epoch")
fig.tight_layout()
plt.show()

In [ ]:
EPOCH = list(ckpts.keys())[1999]  # choose from available checkpoints

alignment, fig, ax = plot_first_layer_odt_alignment(
    state_dict=ckpts[EPOCH],
    tree=tree,
    depth=DEPTH,
    epoch=EPOCH,
    seed=SEED,
)
plt.show()

# Keep these names available for downstream scratch cells.
first_layer_wts = alignment["first_layer_wts"]
odt_wts = alignment["odt_wts"]
row_norms = alignment["row_norms"]
cos = alignment["cos"]
abs_cos = alignment["abs_cos"]
max_abs_cos = alignment["max_abs_cos"]
closest_odt = alignment["closest_odt"]
closest_odt_np = alignment["closest_odt_np"]
node_level = alignment["node_level"]

## Load hidden-layer and output parameters from a given epoch

In [ ]:
# Load architecture-dependent weights at the chosen snapshot epoch.

ALIGNMENT_ABS_THRESHOLD = 0.8
EDGE_ABS_THRESHOLD = 0.020  # tune this; increase if graph is too dense
FIGSIZE = (14, 18)

state = ckpts[EPOCH]
HIDDEN_WEIGHTS = [
    state[f"hidden_layers.{layer_idx}.weight"].detach().cpu()
    for layer_idx in range(len(HIDDEN_DIMS))
]
W0 = HIDDEN_WEIGHTS[0]
Wout = state["output_layer.weight"].detach().cpu()


In [ ]:
max_w, argmax_w = torch.max(W0.abs(), dim=0)
for input_idx in range(W0.shape[1]):
    neuron_idx = int(argmax_w[input_idx])
    print(
        input_idx,
        max_w[input_idx],
        argmax_w[input_idx],
        W0[neuron_idx].norm(),
    )


## Visualise neural net as graph

In [ ]:
# Layered ReLU graph at a chosen snapshot epoch.
ALIGNMENT_ABS_THRESHOLD = 0.8
EDGE_ABS_THRESHOLD = 0.01  # tune this; increase if graph is too dense
FIGSIZE = (14, 18)

# One mask per hidden layer; edit individual entries to select nodes.
NODE_MASK = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]

graph_alignment, visible_edges, graph_fig, graph_ax = plot_layered_relu_graph(
    state_dict=state,
    tree=tree,
    depth=DEPTH,
    epoch=EPOCH,
    node_mask=NODE_MASK,
    alignment_abs_threshold=ALIGNMENT_ABS_THRESHOLD,
    edge_abs_threshold=EDGE_ABS_THRESHOLD,
    figsize=FIGSIZE,
    require_both_edge_endpoints_selected=True,
)
plt.show()

## Computing neuron responsibilities -- i.e. log loss changes to data entering leaf nodes on neuron removal

In [ ]:
checkpoint_model = checkpoint_model_from_state(
    seed_result["out"]["model"],
    ckpts[EPOCH],
)
base_node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]

neuron_responsibility, neuron_responsibility_df, baseline_leaf_metrics = (
    neuron_leaf_responsibility_from_ablation(
        model=checkpoint_model,
        x_eval=seed_result["data"]["x_eval"],
        y_eval=seed_result["data"]["y_eval"],
        tree=seed_result["tree"],
        base_node_mask=base_node_mask,
        min_log_loss_delta=0.1,
        min_relative_log_loss_delta=None,
        min_accuracy_drop=None,
    )
)
neuron_summary = summarize_neuron_responsibility(neuron_responsibility)


In [ ]:
baseline_leaf_metrics

In [ ]:

show(
    neuron_summary,
    scrollY="500px",
    scrollX=True,
    search=True,
    paging=True,
    pageLength=32,
)

In [ ]:
checkpoint_model = checkpoint_model_from_state(
    seed_result["out"]["model"],
    ckpts[EPOCH],
)

# Keep only selected neurons in one layer while retaining every other layer.
MASKED_LAYER_IDX = min(1, len(HIDDEN_DIMS) - 1)
MASKED_NEURON_IDS = [min(9, HIDDEN_DIMS[MASKED_LAYER_IDX] - 1)]
node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]
node_mask[MASKED_LAYER_IDX][:] = False
node_mask[MASKED_LAYER_IDX][MASKED_NEURON_IDS] = True

leaf_accuracy = masked_accuracy_by_odt_leaf(
    model=checkpoint_model,
    node_mask=node_mask,
    x_eval=seed_result["data"]["x_eval"],
    y_eval=seed_result["data"]["y_eval"],
    tree=seed_result["tree"],
)

display(leaf_accuracy)
print(
    "Overall masked accuracy:",
    np.average(
        leaf_accuracy["accuracy"],
        weights=leaf_accuracy["num_samples"],
    ),
)

## Neuron activity breakdown -- neuron activity based on data entering ODT leaf nodes 

In [ ]:

checkpoint_model = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[EPOCH])

acts = activation_tensor_by_layer_neuron_data(
    model=checkpoint_model,
    x_eval=seed_result["data"]["x_eval"],
)
outputs = checkpoint_model(torch.tensor(seed_result["data"]["x_eval"], dtype=torch.float32)).detach().cpu().numpy()
labels = seed_result["data"]["y_eval"]

residues = np.exp(-labels*outputs)/(1+np.exp(-labels*outputs))
leaf_ids = odt_leaf_ids_for_x(seed_result["data"]["x_eval"], seed_result["tree"])

print("activation_tensor shape:", acts.shape)  # (layers, neurons, num_data)



In [ ]:
ACTIVITY_LAYER_IDX = len(HIDDEN_DIMS) - 1
ACTIVITY_NEURON_ID = 0
internal_node_count = tree.w_list.shape[0]
leaf_bins = np.arange(internal_node_count, 2 * internal_node_count + 2)

np.histogram(
    leaf_ids[acts[ACTIVITY_LAYER_IDX, ACTIVITY_NEURON_ID]],
    bins=leaf_bins,
)

In [ ]:
from layer_lenses.relu_analysis import first_layer_neuron_weighted_lens

FIRST_LAYER_NEURON_ID = min(30, HIDDEN_DIMS[0] - 1)
state = ckpts[EPOCH]
checkpoint_model = checkpoint_model_from_state(
    seed_result["out"]["model"],
    state,
)
acts = activation_tensor_by_layer_neuron_data(
    model=checkpoint_model,
    x_eval=seed_result["data"]["x_eval"],
)

# Sum every active path from this first-layer neuron to the output.
total_lens = first_layer_neuron_weighted_lens(
    state_dict=state,
    activation_tensor=acts,
    first_layer_neuron_id=FIRST_LAYER_NEURON_ID,
)
total_lens *= residues * -labels
grad_alt = (
    total_lens @ seed_result["data"]["x_eval"] / len(residues)
)

# This should match the log-loss gradient for the selected first-layer neuron.
grad_check = log_loss_gradients(
    model=checkpoint_model,
    x=seed_result["data"]["x_eval"],
    y=seed_result["data"]["y_eval"],
)
true_grad = grad_check["hidden_layers.0.weight"][
    FIRST_LAYER_NEURON_ID
].numpy()


In [ ]:
from tqdm.std import tqdm

# Decompose the first-layer neuron's lens through one downstream hidden layer.
# A one-hidden-layer model has no downstream hidden layer, so its selected
# first-layer neuron is the sole component.
FIRST_LAYER_NEURON_ID = min(30, HIDDEN_DIMS[0] - 1)
PATH_LAYER_IDX = 1 if len(HIDDEN_DIMS) > 1 else 0
PATH_NODE_IDS = (
    range(HIDDEN_DIMS[PATH_LAYER_IDX])
    if PATH_LAYER_IDX > 0
    else [FIRST_LAYER_NEURON_ID]
)
NUM_GRAD_EPOCHS = 100
analysis_epochs = sorted(ckpts)[-NUM_GRAD_EPOCHS:]
internal_node_count = tree.w_list.shape[0]

for curr_path_node in PATH_NODE_IDS:
    path_node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]
    path_node_mask[PATH_LAYER_IDX][:] = False
    path_node_mask[PATH_LAYER_IDX][curr_path_node] = True

    true_grad_dict = {}
    grad_alt_dict = {}
    W0_dict = {}
    for epoch in tqdm(analysis_epochs):
        state = ckpts[epoch]
        checkpoint_model = checkpoint_model_from_state(
            seed_result["out"]["model"],
            state,
        )
        x_eval = seed_result["data"]["x_eval"]
        labels = seed_result["data"]["y_eval"]
        outputs = checkpoint_model(
            torch.tensor(x_eval, dtype=torch.float32)
        ).detach().cpu().numpy()
        residues = np.exp(-labels * outputs) / (1 + np.exp(-labels * outputs))
        residues = (residues + 1000.0) / 2000.0

        acts = activation_tensor_by_layer_neuron_data(
            model=checkpoint_model,
            x_eval=x_eval,
        )
        total_lens = first_layer_neuron_weighted_lens(
            state_dict=state,
            activation_tensor=acts,
            first_layer_neuron_id=FIRST_LAYER_NEURON_ID,
            node_mask=path_node_mask,
        )
        total_lens *= residues * -labels
        grad_alt = total_lens @ x_eval / len(residues)

        grad_check = log_loss_gradients(
            model=checkpoint_model,
            x=x_eval,
            y=labels,
        )
        true_grad = grad_check["hidden_layers.0.weight"][
            FIRST_LAYER_NEURON_ID
        ].numpy()
        true_grad_dict[epoch] = true_grad
        grad_alt_dict[epoch] = grad_alt
        W0_dict[epoch] = state["hidden_layers.0.weight"][
            FIRST_LAYER_NEURON_ID
        ].numpy()

    true_grad_comps = {}
    alt_grad_comps = {}
    true_grad_comps_numpy = np.zeros(
        (len(true_grad_dict), internal_node_count)
    )
    alt_grad_comps_numpy = np.zeros(
        (len(grad_alt_dict), internal_node_count)
    )
    for row_idx, epoch in enumerate(analysis_epochs):
        true_grad_comps[epoch] = odt_wts.numpy() @ true_grad_dict[epoch]
        true_grad_comps_numpy[row_idx] = true_grad_comps[epoch]
        alt_grad_comps[epoch] = odt_wts.numpy() @ grad_alt_dict[epoch]
        alt_grad_comps_numpy[row_idx] = alt_grad_comps[epoch]

    for odt_node_idx in range(internal_node_count - 1):
        plt.plot(analysis_epochs, alt_grad_comps_numpy[:, odt_node_idx])
    plt.axhline(0.0, color="gray", linewidth=1)
    plt.plot(
        analysis_epochs,
        alt_grad_comps_numpy[:, -1],
        "r--",
    )
    plt.title(
        f"Through node {curr_path_node} in hidden layer {PATH_LAYER_IDX + 1}"
    )
    plt.show()


In [ ]:
from tqdm.std import tqdm

FIRST_LAYER_NEURON_ID = min(30, HIDDEN_DIMS[0] - 1)
PATH_LAYER_IDX = 1 if len(HIDDEN_DIMS) > 1 else 0
SELECTED_PATH_NODE_IDS = (
    [min(34, HIDDEN_DIMS[PATH_LAYER_IDX] - 1)]
    if PATH_LAYER_IDX > 0
    else [FIRST_LAYER_NEURON_ID]
)
NUM_GRAD_EPOCHS = 100
analysis_epochs = sorted(ckpts)[-NUM_GRAD_EPOCHS:]
internal_node_count = tree.w_list.shape[0]

path_node_mask = [np.ones(width, dtype=bool) for width in HIDDEN_DIMS]
path_node_mask[PATH_LAYER_IDX][:] = False
path_node_mask[PATH_LAYER_IDX][SELECTED_PATH_NODE_IDS] = True

true_grad_dict = {}
grad_alt_dict = {}
W0_dict = {}
for epoch in tqdm(analysis_epochs):
    state = ckpts[epoch]
    checkpoint_model = checkpoint_model_from_state(
        seed_result["out"]["model"],
        state,
    )
    x_eval = seed_result["data"]["x_eval"]
    labels = seed_result["data"]["y_eval"]
    outputs = checkpoint_model(
        torch.tensor(x_eval, dtype=torch.float32)
    ).detach().cpu().numpy()
    residues = np.exp(-labels * outputs) / (1 + np.exp(-labels * outputs))
    residues = (residues + 1000.0) / 2000.0

    acts = activation_tensor_by_layer_neuron_data(
        model=checkpoint_model,
        x_eval=x_eval,
    )
    total_lens = first_layer_neuron_weighted_lens(
        state_dict=state,
        activation_tensor=acts,
        first_layer_neuron_id=FIRST_LAYER_NEURON_ID,
        node_mask=path_node_mask,
    )
    total_lens *= residues * -labels
    grad_alt = total_lens @ x_eval / len(residues)

    grad_check = log_loss_gradients(
        model=checkpoint_model,
        x=x_eval,
        y=labels,
    )
    true_grad = grad_check["hidden_layers.0.weight"][
        FIRST_LAYER_NEURON_ID
    ].numpy()
    true_grad_dict[epoch] = true_grad
    grad_alt_dict[epoch] = grad_alt
    W0_dict[epoch] = state["hidden_layers.0.weight"][
        FIRST_LAYER_NEURON_ID
    ].numpy()

true_grad_comps = {}
alt_grad_comps = {}
true_grad_comps_numpy = np.zeros(
    (len(true_grad_dict), internal_node_count)
)
alt_grad_comps_numpy = np.zeros(
    (len(grad_alt_dict), internal_node_count)
)
for row_idx, epoch in enumerate(analysis_epochs):
    true_grad_comps[epoch] = odt_wts.numpy() @ true_grad_dict[epoch]
    true_grad_comps_numpy[row_idx] = true_grad_comps[epoch]
    alt_grad_comps[epoch] = odt_wts.numpy() @ grad_alt_dict[epoch]
    alt_grad_comps_numpy[row_idx] = alt_grad_comps[epoch]


In [ ]:
for j in [0,2,5]:
    plt.plot(alt_grad_comps.keys(),alt_grad_comps_numpy[:,j] )
    plt.plot([min(true_grad_comps.keys()),max(true_grad_comps.keys())],[0,0])
plt.plot(alt_grad_comps.keys(),alt_grad_comps_numpy[:,30],"r--" )
plt.show()

    


In [ ]:
true_grad_dict[1900]/grad_alt_dict[1900]

In [ ]:
true_grad_comps = {}
alt_grad_comps = {}
true_grad_comps_numpy = np.zeros((len(true_grad_dict),31))
alt_grad_comps_numpy = np.zeros((len(grad_alt_dict),31))
for epoch in grad_alt_dict:
    true_grad_comps[epoch] = odt_wts.numpy()@true_grad_dict[epoch]
    true_grad_comps_numpy[epoch-min(true_grad_dict.keys())] = true_grad_comps[epoch]
    alt_grad_comps[epoch] = odt_wts.numpy()@grad_alt_dict[epoch]
    alt_grad_comps_numpy[epoch-min(grad_alt_dict.keys())] = alt_grad_comps[epoch]



In [ ]:

# all paths through node i1 in layer 1
for j in range(30):
    plt.plot(alt_grad_comps.keys(),alt_grad_comps_numpy[:,j] )
    plt.plot([min(true_grad_comps.keys()),max(true_grad_comps.keys())],[0,0])
plt.plot(alt_grad_comps.keys(),alt_grad_comps_numpy[:,30],"r--" )
plt.show()

In [ ]:
plt.plot(true_grad_comps.keys(),true_grad_comps_numpy[:,0] )
plt.plot(alt_grad_comps.keys(),alt_grad_comps_numpy[:,0] )
plt.plot([min(true_grad_comps.keys()),max(true_grad_comps.keys())],[0,0])

In [ ]:
plt.plot(grad_comps.keys(),grad_comps_numpy[:,0])
plt.plot(grad_comps.keys(),grad_comps_numpy[:,2])
plt.plot(grad_comps.keys(),grad_comps_numpy[:,6])
plt.plot(grad_comps.keys(),grad_comps_numpy[:,14])
plt.plot(grad_comps.keys(),grad_comps_numpy[:,30])
plt.plot([min(grad_comps.keys()),max(grad_comps.keys())],[0,0])

In [ ]:
for j in [0,1,2,30]:
    plt.plot(np.cumsum(grad_comps_numpy[100:300,j]))
plt.plot([0,200],[0,0])

In [ ]:
W0_comps = {}
W0_comps_numpy = np.zeros((500,31))
for epoch in range(0,500):
    W0_comps[epoch] = odt_wts.numpy()@W0_dict[epoch]
    W0_comps_numpy[epoch] = W0_comps[epoch]


In [ ]:

plt.plot(W0_comps.keys(),W0_comps_numpy[:,0])
plt.plot(W0_comps.keys(),W0_comps_numpy[:,2])
plt.plot(W0_comps.keys(),W0_comps_numpy[:,6])
plt.plot(W0_comps.keys(),W0_comps_numpy[:,14])
plt.plot(W0_comps.keys(),W0_comps_numpy[:,30])
plt.plot(W0_comps.keys(),W0_comps_numpy[:,24])
plt.plot([0,500],[0,0])

In [ ]:
print(np.max(np.abs(odt_wts.numpy()@grad_alt)))
print(np.argmax(np.abs(odt_wts.numpy()@grad_alt)))
print(odt_wts.numpy()@grad_alt)
print(odt_wts.numpy()@true_grad)

In [ ]:
W0[i1]@odt_wts[30]/(torch.norm(W0[i1])*torch.norm(odt_wts[30]))

In [ ]:
u_ODT_node = seed_result["tree"].w_list[ODT_INTERNAL_NODE_ID]
print("Cosine similarity between grad_alt for neuron i1 and u_ODT_node at epoch", EPOCH, ":", grad_alt@u_ODT_node/(np.linalg.norm(u_ODT_node)*np.linalg.norm(grad_alt)))
print("Norm of grad_alt:", np.linalg.norm(grad_alt))

In [ ]:
from layer_lenses.relu_analysis import summarize_neuron_leaf_activity

checkpoint_model = checkpoint_model_from_state(seed_result["out"]["model"], ckpts[EPOCH])

neuron_leaf_activity_summary = summarize_neuron_leaf_activity(
    model=checkpoint_model,
    x_eval=seed_result["data"]["x_eval"],
    tree=seed_result["tree"],
)

# display(neuron_leaf_activity_summary)
# show(
#     neuron_leaf_activity_summary,
#     scrollY="500px",
#     scrollX=True,
#     search=True,
#     paging=True,
#     pageLength=25,
# )

In [ ]:
# Build: neuron -> activity vector over ODT leaves
# Input: neuron_leaf_activity_summary (DataFrame from earlier cell)
# Output: neuron_to_leaf_activity (dict)

from layer_lenses.relu_analysis import neuron_leaf_activity_dict

neuron_to_leaf_activity, neuron_activity_leaf_order = neuron_leaf_activity_dict(
    neuron_leaf_activity_summary,
    key_mode="layer_local",
)

print(f"Built dictionary for {len(neuron_to_leaf_activity)} neurons.")
print(f"Vector length per neuron: {len(neuron_activity_leaf_order)} leaves")
print("Example key:", next(iter(neuron_to_leaf_activity)))
print("Example vector:", neuron_to_leaf_activity[next(iter(neuron_to_leaf_activity))])

In [ ]:
neuron_to_leaf_activity[(1,7)]

## Start/end metrics

This block computes start/end metrics for the selected seed.

In [ ]:
if seed_result is None:
    raise ValueError("Run the training cell first to populate seed_result.")

rows = collect_start_end_metrics(seed_result)
metrics_table = pd.DataFrame(rows).sort_values(["timepoint"]).reset_index(drop=True)
display(metrics_table)

## Quick visualization

Bar plots for start vs end test error and train log loss for the selected seed.

In [ ]:
if seed_result is None:
    raise ValueError("Run the training cell first to populate seed_result.")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(metrics_table["timepoint"], metrics_table["test_error"])
axes[0].set_title("Test error")

axes[1].bar(metrics_table["timepoint"], metrics_table["train_log_loss"])
axes[1].set_title("Train log loss")

plt.tight_layout()
plt.show()